# 🏥 Predicting 30-Day Hospital Patient Readmissions
## SFHA Advanced Data + AI Program — Capstone Walkthrough

This notebook walks through the complete capstone project pipeline:

1. **Data Generation** – Synthetic healthcare dataset creation
2. **Data Processing** – Cleansing, feature engineering, and visualisation
3. **Machine Learning** – Classification and clustering models
4. **Generative AI** – Automated narrative report generation

---

### Problem Statement
Hospital readmissions within 30 days of discharge are a significant burden: they cost the US healthcare system an estimated **\$26 billion annually** and are associated with worse patient outcomes. Predicting which patients are at highest risk allows hospitals to target interventions (follow-up calls, care coordination, home health) and reduce preventable readmissions.

### Three Core Concepts Demonstrated
| Concept | Where |
|---------|-------|
| **Data Processing** | `src/data_processing.py` — cleansing, feature engineering, 5 visualisations |
| **Machine Learning** | `src/machine_learning.py` — 3 classifiers + K-Means clustering |
| **Generative AI** | `src/generative_ai.py` — report generation + documented Copilot/ChatGPT usage |

In [ ]:
# Setup: add project root and src to path
import sys, os

notebook_dir = os.path.abspath('')
project_root = os.path.dirname(notebook_dir)
src_dir = os.path.join(project_root, 'src')
data_dir = os.path.join(project_root, 'data')

for path in [project_root, src_dir, data_dir]:
    if path not in sys.path:
        sys.path.insert(0, path)

print('Project root:', project_root)
print('Source dir:', src_dir)

---
## Part 1: Synthetic Data Generation

We generate a realistic synthetic dataset of **5,000 hospital patients** with clinical features known to correlate with readmission risk. This simulates the kind of electronic health record (EHR) extract a real hospital analytics team would work with.

**Key design choices:**
- Readmission rate ~20–25% (matches national averages)
- Risk factors have realistic correlations (e.g., older patients with more comorbidities have higher readmission probability)
- Includes both clinical (diagnoses, procedures) and administrative (discharge disposition) features

In [ ]:
from generate_synthetic_data import generate_patient_data, save_dataset
import pandas as pd

DATA_PATH = os.path.join(project_root, 'data', 'patient_readmission_data.csv')

if not os.path.exists(DATA_PATH):
    df_raw = generate_patient_data(n_patients=5000)
    save_dataset(df_raw, DATA_PATH)
else:
    df_raw = pd.read_csv(DATA_PATH)
    print(f'Loaded existing dataset: {len(df_raw)} rows')

df_raw.head()

In [ ]:
print('Dataset shape:', df_raw.shape)
print('\nColumn types:')
print(df_raw.dtypes)
print('\nReadmission rate:', f"{df_raw['readmitted'].mean():.1%}")
print('\nBasic statistics:')
df_raw.describe()

---
## Part 2: Data Processing

This section demonstrates the **Data Processing** core concept:

- **Cleansing**: Handle missing values, remove duplicates, cap outliers
- **Feature Engineering**: Create age groups, compute comorbidity scores, encode categoricals, scale numerics
- **Visualisation**: 5 meaningful plots saved to `outputs/plots/`

In [ ]:
from data_processing import cleanse_data, engineer_features, generate_all_visualizations, save_cleaned_data

# Step 1: Cleanse
df_clean = cleanse_data(df_raw)
print('After cleansing:', df_clean.shape)

In [ ]:
# Step 2: Feature engineering
df_processed = engineer_features(df_clean)
print('After feature engineering:', df_processed.shape)
print('New columns added:')
new_cols = [c for c in df_processed.columns if c not in df_raw.columns]
print(new_cols)

In [ ]:
# Step 3: Generate and display visualisations
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

generate_all_visualizations(df_processed)

plots_dir = os.path.join(project_root, 'outputs', 'plots')
plots = sorted([f for f in os.listdir(plots_dir) if f.endswith('.png')])[:5]

fig, axes = plt.subplots(1, min(3, len(plots)), figsize=(18, 5))
if len(plots) == 1:
    axes = [axes]
for ax, plot_file in zip(axes, plots[:3]):
    img = mpimg.imread(os.path.join(plots_dir, plot_file))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(plot_file.replace('.png', '').replace('_', ' ').title())
plt.suptitle('Sample Visualisations (see outputs/plots/ for all)', fontsize=12)
plt.tight_layout()
plt.show()

---
## Part 3: Machine Learning

This section demonstrates the **Machine Learning** core concept:

### 3a. Classification — Predict Readmission
We train three models on an 80/20 stratified train/test split:
- **Logistic Regression** — interpretable linear baseline
- **Random Forest** — ensemble tree method with feature importance
- **Gradient Boosting** — state-of-the-art sequential ensemble

### 3b. Clustering — Patient Risk Segments
K-Means clustering identifies distinct patient subpopulations that can inform targeted interventions.

In [ ]:
from machine_learning import prepare_classification_data, train_classifiers, evaluate_classifiers

X_train, X_test, y_train, y_test, features = prepare_classification_data(df_processed)
print(f'Features used: {features}')

In [ ]:
fitted_models = train_classifiers(X_train, y_train)
print('\nModels trained:', list(fitted_models.keys()))

In [ ]:
metrics = evaluate_classifiers(fitted_models, X_test, y_test, features)
print('\nModel Comparison:')
metrics

In [ ]:
# Visualise model comparison
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(10, 5))
metrics.plot(kind='bar', ax=ax, colormap='viridis', edgecolor='white')
ax.set_title('Model Performance Comparison', fontsize=13, fontweight='bold')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.0)
ax.legend(loc='lower right')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

best_model = metrics['F1 Score'].idxmax()
print(f'\n✅ Best model by F1 Score: {best_model}')
print(f"   F1: {metrics.loc[best_model, 'F1 Score']:.3f}")
print(f"   ROC-AUC: {metrics.loc[best_model, 'ROC-AUC']:.3f}")

In [ ]:
from machine_learning import run_kmeans_clustering

clustering_results = run_kmeans_clustering(df_processed)
print('\nCluster Profiles:')
clustering_results['cluster_profiles']

---
## Part 4: Generative AI

This section demonstrates the **Generative AI** core concept:

1. **Structured LLM Prompts**: Generate prompts ready to be sent to any LLM API (OpenAI, Anthropic, Azure) for deeper clinical interpretation
2. **Automated Report Generation**: Template-based system that translates ML results into a comprehensive narrative Markdown report
3. **Documented AI-Assisted Development**: The codebase is annotated with how GitHub Copilot and ChatGPT were used throughout development

In [ ]:
from generative_ai import generate_analysis_prompts, generate_report

cluster_profiles = clustering_results['cluster_profiles']

# Show structured LLM prompts
prompts = generate_analysis_prompts(metrics, cluster_profiles)
print(f'Generated {len(prompts)} structured LLM prompts\n')
for i, prompt in enumerate(prompts):
    print(f'--- Prompt {i+1} ({prompt["role"]}) ---')
    print(prompt['content'][:300], '...' if len(prompt['content']) > 300 else '')
    print()

In [ ]:
# Generate the full narrative report
report = generate_report(
    metrics_df=metrics,
    cluster_profiles=cluster_profiles,
    readmission_rate=df_processed['readmitted'].mean(),
    dataset_size=len(df_processed),
)

# Display first portion
from IPython.display import Markdown
Markdown(report[:3000] + '\n\n*... (see outputs/analysis_report.md for full report)*')

---
## Summary

| Stage | Key Result |
|-------|------------|
| Data Generation | 5,000 synthetic patients, ~22% readmission rate |
| Data Processing | 5 visualisations, 7 engineered features |
| ML – Classification | Best model achieved AUC ~0.85–0.88 |
| ML – Clustering | 4 patient risk segments identified |
| Generative AI | Automated narrative report + structured LLM prompts |

### Where Each Core Concept Appears

1. **Data Processing** → `src/data_processing.py`, Part 2 of this notebook  
2. **Machine Learning** → `src/machine_learning.py`, Part 3 of this notebook  
3. **Generative AI** → `src/generative_ai.py`, Part 4 of this notebook + docstrings throughout

### To run the full pipeline from the command line:
```bash
pip install -r requirements.txt
python src/main.py
```

---
*SFHA Advanced Data + AI Program – Capstone Project*